# NB08: Leave-one-cancer-out (LOCO) cancer-disjoint evaluation

Reproduces Supplementary Table S11 and Figures 1D / 4C: leave-one-cancer-
type-out evaluation across TCGA cancer types with at least one HRD-positive
event in the validation+test cohort.

**Manuscript reference** (Methods, Results, Supp Table S11, Figures 1D/4C):
- For each held-out cancer C: train on all patients with non-null scarHRD
  in cancer != C (across all splits); evaluate on val+test of cancer C
- Same pipeline as NB07: StandardScaler -> PCA(384) -> Ridge(alpha=30.0) -> Platt
- 19 cancer types with at least one HRD-positive case
- Headline numbers: BRCA 0.660, LUAD 0.627, LUSC 0.628, CESC 0.502,
  HNSC 0.563, PRAD 0.938 (n=1, exclude), LIHC 0.732, UCEC 0.677, ESCA 0.647
- Histology benchmark lines (Fig 4C):
  - adeno/serous (BRCA + LUAD + STAD), mean = 0.60
  - squamous (HNSC + LUSC + CESC), mean = 0.56

**Required env**: `WORKSPACE`. **Inputs**: clean embeddings + labels.
**Outputs**: `results/loco/per_cancer.csv`, `figures/fig1d_loco_alphabetical.{png,pdf}`,
`figures/fig4c_loco_sorted.{png,pdf}`.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
EMB_DIR = WORKSPACE / "embeddings"
LABELS_DIR = WORKSPACE / "labels"
FIG_DIR = WORKSPACE / "figures"
RESULTS_DIR = WORKSPACE / "results" / "loco"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked params (same as NB07)
SEED = 42
PCA_N = 384
RIDGE_ALPHA = 30.0
HRD_THR = 33
BOOT_N = 200
MIN_POS_REPORT = 5

# histology category lists for Figure 4C benchmark lines
ADENO_SEROUS = ["BRCA", "LUAD", "STAD"]
SQUAMOUS = ["HNSC", "LUSC", "CESC"]

print(f"SEED={SEED}, PCA_N={PCA_N}, RIDGE_ALPHA={RIDGE_ALPHA}, HRD_THR={HRD_THR}")


In [ ]:
# load aligned embeddings + labels with non-null HRD
X = pd.read_parquet(EMB_DIR / "patient_means_clean.parquet").set_index("patient")
labels = pd.read_parquet(LABELS_DIR / "labels.parquet")
labels["patient"] = labels["patient"].astype(str).str.upper().str.slice(0, 12)
labels = labels.drop_duplicates("patient").set_index("patient")

common = sorted(set(X.index) & set(labels.index))
X = X.loc[common]
labels = labels.loc[common].copy()
labels = labels.loc[labels["HRD"].notna()].copy()
X = X.loc[labels.index]
labels["HRD_top20"] = (labels["HRD"] >= HRD_THR).astype(int)

print(f"patients with HRD: {len(labels):,}")
print(f"cancer types     : {labels['cancer'].nunique()}")


In [ ]:
# eligible cancers: at least one HRD-positive event in val+test
eval_mask = labels["split"].isin(["val", "test"])
event_counts = labels.loc[eval_mask].groupby("cancer")["HRD_top20"].agg(["size", "sum"])
event_counts = event_counts.rename(columns={"size": "n_eval", "sum": "n_pos_eval"})
eligible = event_counts.index[event_counts["n_pos_eval"] >= 1].tolist()

print(f"cancers eligible for LOCO (>= 1 HRD+ in val+test): {len(eligible)}")
print(event_counts.loc[eligible].sort_values("n_pos_eval", ascending=False).to_string())


In [ ]:
# run LOCO loop: train on cancer != C (all splits), eval on val+test of C
def fit_pipe(X_arr, y_cont, y_bin):
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("pca", PCA(n_components=PCA_N, random_state=SEED)),
        ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=SEED)),
    ])
    pipe.fit(X_arr, y_cont)
    z = pipe.predict(X_arr).reshape(-1, 1)
    platt = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=SEED)
    platt.fit(z, y_bin)
    return pipe, platt

def predict(pipe, platt, X_arr):
    z = pipe.predict(X_arr).reshape(-1, 1)
    return platt.predict_proba(z)[:, 1]

def boot_ci(y, p, fn, n_boot=BOOT_N, seed=SEED):
    if y.sum() == 0 or y.sum() == len(y):
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    n = len(y)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n:
            continue
        try:
            vals.append(fn(yb, pb))
        except Exception:
            pass
    if not vals:
        return (np.nan, np.nan)
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

rows = []
for c in sorted(eligible):
    tr_idx = labels.index[labels["cancer"] != c]
    te_idx = labels.index[(labels["cancer"] == c) & labels["split"].isin(["val", "test"])]
    if len(te_idx) == 0:
        continue

    X_tr = X.loc[tr_idx].values.astype(np.float32)
    y_tr_cont = labels.loc[tr_idx, "HRD"].astype(float).values
    y_tr_bin = labels.loc[tr_idx, "HRD_top20"].astype(int).values

    X_te = X.loc[te_idx].values.astype(np.float32)
    y_te_bin = labels.loc[te_idx, "HRD_top20"].astype(int).values

    if y_te_bin.sum() == 0 or y_te_bin.sum() == len(y_te_bin):
        rows.append({"cancer": c, "n": int(len(te_idx)),
                     "n_pos": int(y_te_bin.sum()),
                     "n_neg": int(len(y_te_bin) - y_te_bin.sum()),
                     "AUROC": np.nan, "AUROC_lo": np.nan, "AUROC_hi": np.nan,
                     "AP": np.nan, "AP_lo": np.nan, "AP_hi": np.nan,
                     "reason": "single_class"})
        continue

    pipe, platt = fit_pipe(X_tr, y_tr_cont, y_tr_bin)
    p_te = predict(pipe, platt, X_te)
    auc = float(roc_auc_score(y_te_bin, p_te))
    ap = float(average_precision_score(y_te_bin, p_te))
    auc_lo, auc_hi = boot_ci(y_te_bin, p_te, roc_auc_score)
    ap_lo, ap_hi = boot_ci(y_te_bin, p_te, average_precision_score)
    rows.append({
        "cancer": c, "n": int(len(te_idx)),
        "n_pos": int(y_te_bin.sum()),
        "n_neg": int(len(y_te_bin) - y_te_bin.sum()),
        "AUROC": auc, "AUROC_lo": auc_lo, "AUROC_hi": auc_hi,
        "AP": ap, "AP_lo": ap_lo, "AP_hi": ap_hi,
        "reason": "ok",
    })
    print(f"  {c:6s}  n={len(te_idx):4d}  pos={int(y_te_bin.sum()):3d}  AUROC={auc:.3f}")

loco_df = pd.DataFrame(rows).sort_values("AUROC", ascending=False).reset_index(drop=True)
loco_df.to_csv(RESULTS_DIR / "per_cancer.csv", index=False)


In [ ]:
# print summary alongside manuscript Supp S11 reference values
ref = {
    "BRCA": 0.660, "LGG": 0.714, "LUAD": 0.627, "PRAD": 0.938,
    "HNSC": 0.563, "LUSC": 0.628, "BLCA": 0.564, "STAD": 0.518,
    "SKCM": 0.490, "LIHC": 0.732, "CESC": 0.502, "COAD": 0.473,
    "SARC": 0.673, "UCEC": 0.677, "ESCA": 0.647, "PAAD": 1.000,
}
print(f"{'cancer':<8} {'n':>5} {'n_pos':>6} {'AUROC':>7}  {'CI_lo':>6}  {'CI_hi':>6}  {'ms_ref':>7}")
for _, r in loco_df.iterrows():
    rv = ref.get(r["cancer"], np.nan)
    rv_str = f"{rv:.3f}" if not pd.isna(rv) else "n/a"
    auc_str = f"{r['AUROC']:.3f}" if pd.notna(r["AUROC"]) else "n/a"
    lo_str = f"{r['AUROC_lo']:.3f}" if pd.notna(r["AUROC_lo"]) else "n/a"
    hi_str = f"{r['AUROC_hi']:.3f}" if pd.notna(r["AUROC_hi"]) else "n/a"
    print(f"{r['cancer']:<8} {r['n']:>5} {r['n_pos']:>6} {auc_str:>7}  {lo_str:>6}  {hi_str:>6}  {rv_str:>7}")


In [ ]:
# Figure 1D: alphabetical bar chart of LOCO AUROC
plot_df = loco_df[(loco_df["AUROC"].notna())].sort_values("cancer").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7.0, 3.6))
xs = np.arange(len(plot_df))
errlo = (plot_df["AUROC"] - plot_df["AUROC_lo"]).clip(lower=0).values
errhi = (plot_df["AUROC_hi"] - plot_df["AUROC"]).clip(lower=0).values
ax.bar(xs, plot_df["AUROC"].values,
       yerr=np.vstack([errlo, errhi]),
       color=["#777" if pp < MIN_POS_REPORT else "C0" for pp in plot_df["n_pos"]],
       capsize=2.5, edgecolor="black", lw=0.5)
ax.axhline(0.5, color="0.5", lw=0.8, ls=":")
ax.set_xticks(xs)
ax.set_xticklabels(plot_df["cancer"], rotation=45, ha="right", fontsize=8)
ax.set_ylim(0, 1.02)
ax.set_ylabel("LOCO AUROC")
ax.set_title("Figure 1D: leave-one-cancer-out, alphabetical (gray = n_pos < %d)" % MIN_POS_REPORT)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1d_loco_alphabetical.png", dpi=300)
fig.savefig(FIG_DIR / "fig1d_loco_alphabetical.pdf")
plt.show()


In [ ]:
# Figure 4C: sorted bar with adeno/serous and squamous benchmark lines
sortable = loco_df[(loco_df["AUROC"].notna()) & (loco_df["n_pos"] >= MIN_POS_REPORT)].copy()
sortable = sortable.sort_values("AUROC", ascending=False).reset_index(drop=True)

def mean_for(cats):
    sub = sortable[sortable["cancer"].isin(cats)]["AUROC"].dropna()
    return float(sub.mean()) if len(sub) else float("nan")

mean_adeno = mean_for(ADENO_SEROUS)
mean_squam = mean_for(SQUAMOUS)
print(f"adeno/serous mean (BRCA+LUAD+STAD): {mean_adeno:.3f}  (manuscript ref: 0.60)")
print(f"squamous mean (HNSC+LUSC+CESC)   : {mean_squam:.3f}  (manuscript ref: 0.56)")

fig, ax = plt.subplots(figsize=(7.5, 4.0))
xs = np.arange(len(sortable))
errlo = (sortable["AUROC"] - sortable["AUROC_lo"]).clip(lower=0).values
errhi = (sortable["AUROC_hi"] - sortable["AUROC"]).clip(lower=0).values
ax.bar(xs, sortable["AUROC"].values,
       yerr=np.vstack([errlo, errhi]),
       color="C0", edgecolor="black", lw=0.5, capsize=2.5)
labels_x = [f"{c}\n(n={n}, +{p})" for c, n, p in
            zip(sortable["cancer"], sortable["n"], sortable["n_pos"])]
ax.set_xticks(xs)
ax.set_xticklabels(labels_x, rotation=45, ha="right", fontsize=7)
ax.axhline(0.5, color="0.5", lw=0.8, ls=":")
if not np.isnan(mean_adeno):
    ax.axhline(mean_adeno, color="C0", lw=1.0, ls="--",
               label=f"adeno/serous mean = {mean_adeno:.2f}")
if not np.isnan(mean_squam):
    ax.axhline(mean_squam, color="C3", lw=1.0, ls="--",
               label=f"squamous mean = {mean_squam:.2f}")
ax.set_ylabel("LOCO AUROC")
ax.set_ylim(0, 1.02)
ax.set_title("Figure 4C: LOCO sorted, with histology means")
ax.legend(loc="upper right", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4c_loco_sorted.png", dpi=300)
fig.savefig(FIG_DIR / "fig4c_loco_sorted.pdf")
plt.show()


In [ ]:
# final report
report = {
    "seed": SEED,
    "params": {"PCA_N": PCA_N, "RIDGE_ALPHA": RIDGE_ALPHA, "HRD_THR": HRD_THR},
    "n_cancers_evaluated": int(len(loco_df)),
    "histology_means": {"adeno_serous": mean_adeno, "squamous": mean_squam},
    "histology_lists": {"adeno_serous": ADENO_SEROUS, "squamous": SQUAMOUS},
    "manuscript_refs": {
        "n_cancers": 19,
        "adeno_serous_mean": 0.60,
        "squamous_mean": 0.56,
        "selected": {
            "BRCA": 0.660, "LUAD": 0.627, "LUSC": 0.628,
            "CESC": 0.502, "HNSC": 0.563, "STAD": 0.518,
            "LIHC": 0.732, "UCEC": 0.677, "ESCA": 0.647,
        },
    },
    "loco_path": str(RESULTS_DIR / "per_cancer.csv"),
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
